# Assignment Data Mining 2026 - Detection of Fake Hotel Reviews
> Reminder: the final submission must be `.py` scripts + `README.md`, not a notebook. Use this notebook for exploring, then move the code into scripts (e.g. `preprocessing.py`, `train_models.py`, `evaluate.py`).

In [ ]:
RANDOM_SEED = 42

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 200)

### The Data
rewiew format: ,Review_Language,City Name,Hotel Name,Upside_Review,Downside_Review,Review_Score,Sentiment,source,Prompt_Language,na_up_review,na_down_review

example: 55,English,Ankara,Esila Hotel,The location was very central and the staff was nice and helped us.,The room smelled like cigarettes and there was mold in the bathroom,4.0,NEG,0,,False,False

Columns in 'all_data.csv' that matter for us:
-'Review_Language': keep only 'English'
-'Upside_Review','Downside_Review': the review text is split in a positive and a negative part (either can be epty/NaN)
- 'Semiment': either 'POS' or 'NEG' (question 3)
- 'source': the class label (1 = AI, 0 = Human). We need 500 positivie and 500 negative per class
- 'Prompt_Language': only filled for AI reviews

In [ ]:
### Load data
df = pd.read_csv("data/all_data.csv", index_col=0)   # first column is just a row index
df = df[df["Review_Language"] == "English"].reset_index(drop=True)

print(df.shape)
print(pd.crosstab(df["source"], df["Sentiment"], margins=True))
print("\nMissing values per column:")
print(df.isna().sum())

### Pre-processing
Only the review **text** is used as input. `Review_Score`, `City Name`, `Hotel Name` and `Prompt_Language` are not used as features: the score is not part of the text, and `Prompt_Language` exists only for AI reviews (it would leak the label).


In [ ]:
### Missing text per class
missing = (
    df.assign(
        up_missing=df["Upside_Review"].isna(),
        down_missing=df["Downside_Review"].isna(),
        both_missing=df["Upside_Review"].isna() & df["Downside_Review"].isna(),
    )
    .groupby(["source", "Sentiment"])[["up_missing", "down_missing", "both_missing"]]
    .mean()
    .round(3)
)
print(missing)

# The na_* flags in the csv do not perfectly match the NaNs.  We use the NaNs themselves
print("\nMismatch na_up_review vs NaN:  ", (df["Upside_Review"].isna() != df["na_up_review"]).sum())
print("Mismatch na_down_review vs NaN:", (df["Downside_Review"].isna() != df["na_down_review"]).sum())

In [ ]:
### Most freq texts per class
for col in ["Upside_Review", "Downside_Review"]:
    print(f"\n{col} - most frequent texts per class")
    for s, name in [(0, "Human"), (1, "AI")]:
        top = df.loc[df["source"] == s, col].str.strip().str.lower().value_counts().head(5)
        print(f"  {name}: {top.to_dict()}")

In [ ]:
###Build one text column per review: concatenate upside and downside part. If missing part, use empty string
df["text_raw"] = (
    df["Upside_Review"].fillna("").str.strip() + " " + df["Downside_Review"].fillna("").str.strip()
).str.strip()

print("Empty reviews:     ", (df["text_raw"] == "").sum())
print("Duplicate reviews: ", df["text_raw"].duplicated().sum())

In [ ]:
###Review length per class / sentiment
df["n_words"] = df["text_raw"].str.split().str.len()
print(df.groupby(["source", "Sentiment"])["n_words"].describe().round(1))

fig, ax = plt.subplots(figsize=(7, 4))
for s, name in [(0, "Human"), (1, "AI")]:
    ax.hist(df.loc[df["source"] == s, "n_words"], bins=np.arange(0, 300, 10), alpha=0.6, label=name)
ax.set_xlabel("Number of words per review")
ax.set_ylabel("Count")
ax.set_title("Review length by class")
ax.legend()
plt.show()

In [ ]:
###Surface cues that differ between classes 
def mentions(col):
    return pd.Series([str(n).lower() in t.lower() for n, t in zip(df[col], df["text_raw"])])

checks = pd.DataFrame({
    "has_digit":           df["text_raw"].str.contains(r"\d"),
    "has_newline":         df["text_raw"].str.contains("\n"),
    "curly_apostrophe":    df["text_raw"].str.contains("’"),
    "straight_apostrophe": df["text_raw"].str.contains("'"),
    "non_ascii":           df["text_raw"].str.contains(r"[^\x00-\x7f]"),
    "mentions_hotel":      mentions("Hotel Name"),
    "mentions_city":       mentions("City Name"),
})
print(checks.groupby(df["source"]).mean().T.rename(columns={0: "Human", 1: "AI"}).round(3))

#### Text cleaning choices
- **Lowercasing**: "Location" and "location" become the same term.
- **Apostrophes normalised** 
- **Hotel and city names removed**: 
- **Accents stripped** 
- **Numbers → `num` token**: We keep this stylistic signal but merge all numbers into one term to avoid many sparse terms.
- **Punctuation, emoji and other non-letter characters removed**: the bag-of-words model only uses words.
- **Stop words are kept**: function words and pronouns ("i", "we", "the", "very") are known cues for deceptive or generated text. Sparse-term removal and feature selection come later and are done on the training set only.
- **No stemming/lemmatization**: GPT-4 tends to repeat specific word forms, and keeping them makes the most important features easier to read.

In [ ]:
### Text cleaning
import unicodedata

REMOVE_NAMES = True  #switch to remove hotels and city names so we can check the impact of this change

def preprocess(text, hotel=None, city=None):
    text = text.lower()
    text = text.replace("’", "'").replace("‘", "'")   #curly to straight apostrophe
    if REMOVE_NAMES:
        for name in (hotel, city):
            if isinstance(name, str) and name:
                text = text.replace(name.lower(), " ")
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()   #strip accents
    text = re.sub(r"\d+(?:[.,]\d+)*", " num ", text)      # numbers -> 'num'
    text = re.sub(r"[^a-z' ]+", " ", text)             # keep letters and apostrophes
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = [preprocess(t, h, c) for t, h, c in zip(df["text_raw"], df["Hotel Name"], df["City Name"])]

print("Empty after cleaning:", (df["text"] == "").sum())
df.sample(5, random_state=RANDOM_SEED)[["source", "Sentiment", "text_raw", "text"]]

In [ ]:
###objects used in the rest of the notebook
X_text    = df["text"]
y         = df["source"].astype(int)     # 1=AI , 0=human 
sentiment = df["Sentiment"]              # needed for q3

print(X_text.shape, y.value_counts().to_dict())

### Splitting strategy

1- Split the data randomly into a training/validation set (75%) and a test set(25%). 
2- Use stratified random sampling so that the class distribution in the training set and test set are the same.

The test set (500 reviews) is only used at the very end, to estimate the performance of the five final models. Everything else (vectorizer vocabulary, sparse-term removal, feature selection, hyperparameter tuning) is fitted on the training set only.

In [ ]:
### TODO: Stratified 75/25 split
from sklearn.model_selection import train_test_split

###Bag-of-words representation
1- Represent the review texts using word frequencies (bag-of-words representation).

In [ ]:
### TODO: Bag-of-words
from sklearn.feature_extraction.text import CountVectorizer

## Classifiers and hyperparameter tuning

Use the training/validation set (1500 reviews) for training and hyperparameter selection with cross-validation or (for random forests) out-of-bag evaluation. After tuning, train each model on the complete training set with the selected hyperparameter values.

With hyperparameters we mean parameters that influence the complexity of the model. Save the tuning results (e.g. CV score per value) so you can put them in a table/figure in the appendix of the report.

#### 1. Multinomial naive Bayes (generative linear classifier)
Hyperparameter: the value of *k* in selecting the top-*k* features. Feature selection must happen inside the cross-validation (otherwise the CV score is optimistic).

In [ ]:
### TODO: Multinomial naive Bayes + top-k feature selection


#### 2. Logistic regression with Lasso penalty (discriminative linear classifier)
`penalty="l1"`, and remember that `C = 1/λ`. Consider `LogisticRegressionCV`.

In [ ]:
### TODO: logistic regression


#### 3. Single classification tree (non-linear classifier)
Hyperparameter: the cost-complexity pruning parameter α (`ccp_alpha`). Hint: `cost_complexity_pruning_path` gives you candidate values of α.

In [ ]:
### TODO: Classification tree

#### 4. Random forests (ensemble of trees)
Hyperparameters: number of trees, number of randomly selected features. Tune with out-of-bag evaluation.

In [ ]:
### TODO: Random forest with OOB tuning

#### 5. Gradient boosting (ensemble of trees)
Hyperparameters: shrinkage (learning rate), tree depth, number of trees.

In [ ]:
### TODO: Gradient boosting

### Evaluation on the test set

To measure predictive performance, use accuracy, precision, recall and the F1 score. Decide which class is the "positive" class for precision/recall (fake = 1) and state it in the report.

In [ ]:
###TODO: Quantitative evaluation

### Statistical tests: McNemar
To test whether the difference in accuracy between two classifiers is significant, use McNemar's test (Dietterich, 1998), with significance level α = 0.01. Describe the test in the report and give the relevant 2-by-2 table for each application. You only need a limited number of tests to answer the questions, not all pairwise comparisons.

In [ ]:
### TODO: McNemar's test

## Address the following questions

### Question 1

How does the performance of the generative linear model (multinomial naive Bayes) compare to the discriminative linear model (regularized logistic regression)?

In [ ]:
### Q1

### Question 2

Do the tree ensembles improve on the performance of the single tree?

In [ ]:
### Q2

### Question 3

For the random forest classifier finally selected, is it better at distinguishing between fake and genuine positive reviews or between fake and genuine negative reviews?

Perform an appropriate statistical test to determine whether the difference in accuracy between positive and negative reviews is significant.

In [ ]:
### Q3

### Question 4

For each algorithm, what are the 5 most important features in the final model? For the linear models, make a distinction between features pointing towards an AI-generated review and features pointing towards a review written by a human. Interpret the results.


In [ ]:
### Q4